# SFT LoRA — Qwen3.5-4B (сервер)

Полное обучение на `seed_42` (train + eval) с длинным контекстом до 23k токенов.

In [1]:
import torch
from pathlib import Path
import json
import os

os.environ["HF_HOME"] = "/shared/hf_cache"

ROOT = Path("/workdir")
MODEL_ID = "Qwen/Qwen3.5-4B"
ADAPTER_NAME = "lora_4b_bf16_big_data_12k"
OUTPUT_DIR = ROOT / "output" / ADAPTER_NAME
SEED_DIR = ROOT / "llm_fc_hypernym_prediction/output/dataset/big_data/seed_42"
TRAIN_LIMIT = None
EVAL_LIMIT = None
MAX_LEN = 12000
TRUNCATE_OVERSIZED = True

checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*")) if OUTPUT_DIR.exists() else []
RESUME_FROM = str(checkpoints[-1]) if checkpoints else None

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SEED_DIR:", SEED_DIR)
print("MODEL_ID:", MODEL_ID)
print("MAX_LEN:", MAX_LEN)
print(f"Найдено чекпоинтов: {len(checkpoints)}")
if RESUME_FROM:
    print(f"→ Resume из чекпоинта: {RESUME_FROM}")
else:
    print("→ Чекпоинтов нет, стартуем с нуля")

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


torch: 2.6.0+cu124
cuda: True NVIDIA A100-SXM4-80GB
OUTPUT_DIR: /workdir/output/lora_4b_bf16_big_data_12k
SEED_DIR: /workdir/llm_fc_hypernym_prediction/output/dataset/big_data/seed_42
MODEL_ID: Qwen/Qwen3.5-4B
MAX_LEN: 12000
Найдено чекпоинтов: 0
→ Чекпоинтов нет, стартуем с нуля


# Загрузка и первичный анализ данных

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


train_records_raw = load_jsonl(SEED_DIR / "train.jsonl")
eval_records_raw = load_jsonl(SEED_DIR / "eval.jsonl")
print(f"train: {len(train_records_raw)}, eval: {len(eval_records_raw)}")

train: 11088, eval: 1222


# Туллзы и токенизатор

In [3]:
import copy
import sys

PROJ_ROOT = ROOT / "llm_fc_hypernym_prediction"
sys.path.insert(0, str(PROJ_ROOT))
from utils.tools import hyponym_only

TOOLS = hyponym_only


def normalize_messages_for_template(messages):
    msgs = copy.deepcopy(messages)
    for msg in msgs:
        if msg.get("role") != "assistant":
            continue
        for tc in msg.get("tool_calls") or []:
            fn = tc.get("function", tc)
            args = fn.get("arguments")
            if isinstance(args, str):
                fn["arguments"] = json.loads(args)
    return msgs

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side="left")

def count_tokens(messages: list[dict]) -> int:
    normalized = normalize_messages_for_template(messages)
    result = tokenizer.apply_chat_template(
        normalized,
        tools=TOOLS,
        tokenize=True,
        return_dict=True,
        enable_thinking=False,
    )
    return len(result["input_ids"])

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Подготовка датасета (фильтрация по длине)

In [5]:
def filter_by_length(records: list[dict], limit: int | None) -> tuple[list[dict], int]:
    kept: list[dict] = []
    skipped = 0
    if TRUNCATE_OVERSIZED:
        kept = records[:limit] if limit is not None else records
        return kept, len(records) - len(kept)
    
    for record in records:
        n = count_tokens(record["messages"])
        if n <= MAX_LEN:
            kept.append(record)
        else:
            skipped += 1
        if limit is not None and len(kept) >= limit:
            break
    return kept, skipped


train_filtered, train_skipped = filter_by_length(train_records_raw, TRAIN_LIMIT)
eval_filtered, eval_skipped = filter_by_length(eval_records_raw, EVAL_LIMIT)

print(
    f"train: kept={len(train_filtered)}, skipped_by_len={train_skipped}, "
    f"limit={TRAIN_LIMIT}, max_len<={MAX_LEN}"
)
print(
    f"eval: kept={len(eval_filtered)}, skipped_by_len={eval_skipped}, "
    f"limit={EVAL_LIMIT}, max_len<={MAX_LEN}"
)

train: kept=11088, skipped_by_len=0, limit=None, max_len<=12000
eval: kept=1222, skipped_by_len=0, limit=None, max_len<=12000


# Формирование HuggingFace Dataset

In [6]:
from datasets import Dataset

records_for_train = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in train_filtered
]
records_for_eval = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in eval_filtered
]

ds_train = Dataset.from_list(records_for_train)
ds_eval = Dataset.from_list(records_for_eval)
print("ds_train:", len(ds_train), "ds_eval:", len(ds_eval))

ds_train: 11088 ds_eval: 1222


# Загрузка модели Qwen3.5-4B

In [8]:
import gc
from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation='flash_attention_2',
)
model.config.use_cache = False

base_mem = torch.cuda.memory_allocated() / 1e9
print(f"Модель загружена. Занято памяти: {base_mem:.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 8019.70it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:01<00:00, 366.73it/s]


Модель загружена. Занято памяти: 8.41 GB


# LoRA конфигурация

In [9]:
from peft import LoraConfig, get_peft_model, PeftModel

if RESUME_FROM:
    model = PeftModel.from_pretrained(model, RESUME_FROM)
    print(f"LoRA загружен из чекпоинта: {RESUME_FROM}")
else:
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

model.enable_input_require_grads()
model.print_trainable_parameters()

lora_mem = torch.cuda.memory_allocated() / 1e9
print(f"После LoRA: {lora_mem:.2f} GB (+{lora_mem - base_mem:.2f} GB)")

trainable params: 1,572,864 || all params: 4,207,324,160 || trainable%: 0.0374
После LoRA: 8.42 GB (+0.01 GB)


# Обучение LoRA + SFT

In [ ]:
from trl import SFTConfig, SFTTrainer

gc.collect()
torch.cuda.empty_cache()

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    bf16=True,
    logging_steps=1,
    save_steps=10,
    max_length=MAX_LEN,
    truncation_mode="keep_start",
    assistant_only_loss=True,
    gradient_checkpointing=True,
    loss_type="chunked_nll",
    report_to="tensorboard",

    eval_strategy="steps",
    eval_steps=10,
    # eval_on_start=True,
    per_device_eval_batch_size=2,

    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

print(f"train: {len(trainer.train_dataset)}, eval: {len(trainer.eval_dataset)}")
print(f"checkpoints → {OUTPUT_DIR}")

[2026-07-09 14:25:03,801] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
Building labels for eval dataset: 100%|██████████| 1222/1222 [00:12<00:00, 94.96 examples/s] 

train: 11088, eval: 1222
checkpoints → /workdir/output/lora_4b_bf16_big_data_12k


In [11]:
trainer.train(resume_from_checkpoint=RESUME_FROM)
trainer.save_model(str(OUTPUT_DIR))
print(f"Финальный адаптер сохранён: {OUTPUT_DIR}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 